# **Install**

In [1]:
!pip install transformers seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=f7c9983aa9b07b24bf946433f1429cd33da690e6a66445546b8577d45fe91554
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


# **Create dataset**

In [2]:
texts = [
    ["John", "works", "at", "Google"],
    ["Mary", "lives", "in", "London"],
    ["Apple", "is", "a", "company"],
    ["He", "is", "reading", "a", "book"]
]

labels = [
    ["B-PER", "O", "O", "B-ORG"],
    ["B-PER", "O", "O", "B-LOC"],
    ["B-ORG", "O", "O", "O"],
    ["O", "O", "O", "O", "O"]
]

# **Label Mapping**

In [3]:
unique_labels = list(set([l for sub in labels for l in sub]))

label2id = {l: i for i, l in enumerate(unique_labels)}
id2label = {i: l for l, i in label2id.items()}

# **Tokenizer**

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# **Tokenization + Label Alignment**

In [5]:
def tokenize_and_align(texts, labels):
    tokenized = tokenizer(
        texts,
        is_split_into_words=True,
        padding=True,
        truncation=True
    )

    new_labels = []

    for i, label in enumerate(labels):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)

            prev = word_idx

        new_labels.append(label_ids)

    tokenized["labels"] = new_labels
    return tokenized

In [6]:
encodings = tokenize_and_align(texts, labels)

# **Dataset Class**

In [7]:
import torch

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

In [8]:
dataset = CustomDataset(encodings)

# **Model Setup**

In [9]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# **Training**

In [10]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [11]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=1.2356173197428386, metrics={'train_runtime': 5.8421, 'train_samples_per_second': 2.054, 'train_steps_per_second': 1.027, 'total_flos': 21436049376.0, 'train_loss': 1.2356173197428386, 'epoch': 3.0})

# **EValuation**

In [12]:
from seqeval.metrics import classification_report

# dummy prediction (since small dataset)
predictions = trainer.predict(dataset)

import numpy as np

preds = np.argmax(predictions.predictions, axis=2)

true_labels = []
true_preds = []

for i in range(len(preds)):
    pred_labels = []
    true_lab = []

    for j in range(len(preds[i])):
        if encodings["labels"][i][j] != -100:
            pred_labels.append(id2label[preds[i][j]])
            true_lab.append(id2label[encodings["labels"][i][j]])

    true_preds.append(pred_labels)
    true_labels.append(true_lab)

print(classification_report(true_labels, true_preds))

              precision    recall  f1-score   support

         LOC       0.00      0.00      0.00         1
         ORG       0.00      0.00      0.00         2
         PER       0.00      0.00      0.00         2

   micro avg       0.00      0.00      0.00         5
   macro avg       0.00      0.00      0.00         5
weighted avg       0.00      0.00      0.00         5



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# **Inference**

In [13]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

sentence = "John works at Google"

result = nlp(sentence)

for r in result:
    print(r["word"], "→", r["entity"])

**Comparison**

POS Tagging assigns grammatical roles like noun, verb.
Chunking groups words into phrases like noun phrase.

**Challenges**

Handling subword tokenization was difficult.
Aligning labels with tokens required careful processing.
Special tokens were handled using -100.


**Observations**

BERT performs well in capturing context.
Even with small data, model can learn patterns.
More data improves accuracy.
